# Lab 4: Foreign Language to English Translation

**Albert Kabore**  
PhD Student in Artificial Intelligence  
University of the Cumberlands

**Implementation direction: English → Romanian.** The assignment title says “Foreign Language to English,” but Parts A and B explicitly require English to a foreign language. This lab follows those detailed instructions and the supplied English-to-Romanian reference notebook.

**Model:** `google-t5/t5-small` (the reference's `t5-small` checkpoint; pre-trained encoder–decoder Transformer) · **Data:** WMT16 `ro-en`, test split, 1,999 sentence pairs · **Metric:** BLEU (SacreBLEU).

**Contents:** Introduction · Part A – explanation · Part B – implementation (tasks 1–5) · Part C – analysis · Final Insights · Recommendations · Conclusion · References.

## Introduction

Machine translation provides a practical setting for studying how artificial intelligence represents language, generates text, and measures output quality. This lab implements English-to-Romanian translation with a pre-trained T5 model and evaluates its predictions against human references from the WMT16 test set. The implementation follows the chapter 15 notebook supplied with the assignment (Surdeanu & Valenzuela-Escárcega, 2024).

The objective is to explain the translation process, provide executable commands, and interpret the resulting BLEU scores. The workflow covers library initialization, tokenizer and model loading, dataset preparation, batch translation, and evaluation. It uses existing model weights without additional training. Beyond the primary greedy-decoding result, optional diagnostics examine Romanian diacritics, sentence-level variation, and beam-search costs. The analysis connects quantitative results with concrete translation examples to identify limitations and priorities for future improvement.

## Part A: Machine Translation (English → Foreign Language)

Machine translation converts text from a source language into a target language while preserving meaning. This lab translates English into Romanian with T5, a pre-trained neural sequence-to-sequence model.

**Architecture.** A Transformer encoder reads the English tokens and uses self-attention to produce representations that depend on context. A decoder generates Romanian tokens sequentially, attending to previously generated tokens and the encoder's representations. These attention mechanisms help the model learn word relationships and reorder phrases across languages (Vaswani et al., 2017).

**Tokenization.** The tokenizer splits text into SentencePiece subword units and maps them to integer IDs. Common words may use one token; rare words and inflections may require several. Subwords reduce vocabulary size, but unsupported characters can still become unknown tokens. T5 uses approximately 32,000 subword pieces; this checkpoint's tokenizer reports 32,100 entries including special tokens.

**Model selection.** Training translation models requires aligned sentence pairs and substantial computation. Instead, this implementation loads `google-t5/t5-small` with `AutoTokenizer` and `AutoModelForSeq2SeqLM`. T5 represents tasks as text-to-text problems, and its training mixture included English-to-Romanian translation. Prefixing English input with `translate English to Romanian: ` selects that learned task (Raffel et al., 2020). This lab uses existing weights without additional training.

**Implementation.** Import PyTorch, NumPy, and Transformers, set random seeds, and choose a CPU or available GPU. Load the tokenizer and model, move the model to the selected device, and call `model.eval()` to disable dropout. Load the WMT16 `ro-en` test split with Hugging Face's `load_dataset`. Its English sentences are inputs; its Romanian translations are evaluation references and are never supplied to the model.

Define `translate` to add the task prefix, tokenize batches with padding and truncation, move tensors to the device, call `model.generate`, and decode generated IDs into text. Apply it across the dataset using `Dataset.map`. Gradient tracking is disabled because inference does not update weights. Greedy decoding selects the most probable next token; optional beam search explores several candidates at additional computational cost. Generation ends at an end-of-sequence token or the configured length limit.

**Evaluation.** Compare predictions with human Romanian references using corpus BLEU. BLEU combines clipped 1- through 4-gram precisions with a brevity penalty; SacreBLEU reports it on a 0–100 scale with standardized tokenization (Papineni et al., 2002; Post, 2018). Higher scores indicate greater reference overlap under the same evaluation conditions, rather than percentage accuracy.

Romanian inflection, ambiguous words, idioms, and missing character coverage can cause errors. BLEU also penalizes valid paraphrases, spelling variants, and different word orders. Therefore, inspect example translations alongside the score to assess meaning and fluency.

## Part B: Model Implementation

**Environment.** Python 3.11 with PyTorch, NumPy, Transformers, Datasets, SacreBLEU, SentencePiece, pandas, and Jupyter. From the project folder, install with:

```bash
python -m pip install -r requirements.txt
```

If submitting or using the notebook alone, the equivalent installation command is:

```bash
python -m pip install "torch>=2.2" "numpy>=1.26" "transformers>=4.40" "datasets>=2.19" "sacrebleu>=2.4" "sentencepiece>=0.2" "pandas>=2.2" "jupyter>=1.0"
```

Run code cells from top to bottom. The saved greedy run took 496 seconds (about 8 minutes) on CPU; the optional beam-search extension took an additional 4,085 seconds (about 68 minutes). Timings depend on hardware and caching. The code uses a GPU when available. `N_SAMPLES = None` evaluates all 1,999 pairs; a smaller sample changes the evaluation conditions and scores.

### B.1 Initialize the code to enable torch, numpy, and transformers

Import the libraries, select the compute device, set seeds to support reproducibility, and define the run configuration. Seeds do not guarantee identical results across all hardware and library versions.

In [1]:
import random
import time

import numpy as np
import torch
import transformers
from transformers import set_seed

# Reproducibility: same seed as the reference notebook (seed = 42).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# Use a GPU when present; otherwise fall back to CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Run configuration -- identical to the reference notebook (gentlenlp, chapter 15).
MODEL_NAME = "google-t5/t5-small"  # reference uses "t5-small", a Hub redirect to this same checkpoint
SOURCE_LANG, TARGET_LANG = "en", "ro"
TASK_PREFIX = "translate English to Romanian: "
MAX_SOURCE_LENGTH = 1024            # truncation limit for English inputs (tokens)
MAX_TARGET_LENGTH = 128             # max length of generated Romanian output (tokens)
NUM_BEAMS = 1                       # 1 = greedy decoding (reference setting)
BATCH_SIZE = 100
N_SAMPLES = None                    # None = full test set (1,999 pairs); e.g. 200 for a quick run

print(f"torch {torch.__version__} | numpy {np.__version__} | transformers {transformers.__version__}")
print(f"device: {device} | random seed: {SEED}")

torch 2.14.0+cpu | numpy 2.3.4 | transformers 5.18.0
device: cpu | random seed: 42


### B.2 Load tokenizer and pre-trained model

`AutoTokenizer` and `AutoModelForSeq2SeqLM` read the checkpoint configuration and construct matching tokenizer and model classes. The saved run reports `T5Tokenizer` and `T5ForConditionalGeneration`; class naming can vary by library version. `model.eval()` disables dropout, while `torch.inference_mode()` below disables gradient tracking. These serve different purposes (Hugging Face, T5 documentation).

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{type(tokenizer).__name__} | vocab size: {tokenizer.vocab_size:,}")
print(f"{type(model).__name__} | parameters: {n_params/1e6:.1f}M")

T5Tokenizer | vocab size: 32,100
T5ForConditionalGeneration | parameters: 60.5M


In [3]:
# Peek at how the tokenizer splits an English sentence into subword pieces.
example = TASK_PREFIX + "The weather is beautiful today."
print(tokenizer.tokenize(example))
print(tokenizer(example).input_ids)

['▁translate', '▁English', '▁to', '▁Romania', 'n', ':', '▁The', '▁weather', '▁is', '▁beautiful', '▁today', '.']
[13959, 1566, 12, 3871, 29, 10, 37, 1969, 19, 786, 469, 5, 1]


### B.3 Load dataset from Hugging Face

Use the WMT16 Romanian–English news corpus (Bojar et al., 2016), configuration `ro-en`, official **test** split (`newstest2016`, 1,999 pairs). Each `translation` dictionary contains English (`en`) and a human Romanian reference (`ro`). The configuration name describes the language pair; the keys selected in the code determine the translation direction.

The reference calls `load_dataset("wmt16", "ro-en", split="test")`. This lab uses the namespaced Hub identifier `wmt/wmt16`, listed on the Hugging Face dataset card. Only English is sent to the model; Romanian references are retained for scoring. No fitting or parameter updates use the test data.

In [4]:
from datasets import load_dataset

# Reference notebook: load_dataset('wmt16', 'ro-en', split='test')  -> same data, namespaced id
dataset = load_dataset("wmt/wmt16", "ro-en", split="test")
if N_SAMPLES is not None:
    dataset = dataset.select(range(N_SAMPLES))

print(dataset)
for row in dataset.select(range(3)):
    print("EN:", row["translation"]["en"])
    print("RO:", row["translation"]["ro"])
    print()

Dataset({
    features: ['translation'],
    num_rows: 1999
})
EN: UN Chief Says There Is No Military Solution in Syria
RO: Șeful ONU declară că nu există soluții militare în Siria

EN: Secretary-General Ban Ki-moon says his response to Russia's stepped up military support for Syria is that "there is no military solution" to the nearly five-year conflict and more weapons will only worsen the violence and misery for millions of people.
RO: Secretarul General Ban Ki-moon afirmă că răspunsul său la suportul militar al Rusiei pentru Siria este că „nu există o soluție militară” la conflictul care durează de aproape cinci ani iar mai multe arme nu ar face decât să agraveze violența și suferința a milioane de oameni.

EN: The U.N. chief again urged all parties, including the divided U.N. Security Council, to unite and support inclusive negotiations to find a political solution.
RO: Șeful ONU a solicitat din nou tuturor părților, inclusiv Consiliului de securitate ONU divizat să se unifice și 

### B.4 Implement the translate method and apply it on the dataset

`translate` works on a **batch** of rows, as in the reference notebook. It adds the T5 task prefix to each English sentence and tokenizes the batch with padding and truncation. It then calls `model.generate` (greedy decoding, `num_beams=1`) and decodes the token IDs back to text. It returns the human `reference` and the model `prediction`. `Dataset.map(batched=True, remove_columns=...)` applies it to all 1,999 rows in batches of 100.

In [5]:
@torch.inference_mode()  # no gradient tracking -> less memory, faster
def translate(batch, num_beams=NUM_BEAMS):
    """Translate a batch of WMT16 rows from English to Romanian.

    Args:
        batch: dict with key "translation" -> list of {"en": str, "ro": str}.
        num_beams: 1 = greedy decoding; >1 = beam search.
    Returns:
        dict with "reference" (human Romanian) and "prediction" (model Romanian) lists.
    """
    sources = [TASK_PREFIX + pair[SOURCE_LANG] for pair in batch["translation"]]
    targets = [pair[TARGET_LANG] for pair in batch["translation"]]
    inputs = tokenizer(
        sources,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,       # cut inputs longer than MAX_SOURCE_LENGTH
        padding=True,          # pad to the longest sentence in the batch
        return_tensors="pt",
    ).to(device)
    output_ids = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,   # ignore padding positions
        num_beams=num_beams,
        do_sample=False,       # explicitly disable sampling for greedy/beam decoding
        max_length=MAX_TARGET_LENGTH,
    )
    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    return {"reference": targets, "prediction": decoded}


# Sanity check on the reference notebook's example sentence.
print(translate({"translation": [{"en": "this is a test", "ro": ""}]})["prediction"][0])

Acesta este un test


In [6]:
start = time.perf_counter()
results = dataset.map(
    translate,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=dataset.column_names,   # keep only reference + prediction
    desc="Translating (greedy)",
)
elapsed = time.perf_counter() - start
print(f"Translated {len(results):,} sentences in {elapsed:.0f}s "
      f"({len(results)/elapsed:.1f} sentences/s on {device})")
results.to_pandas().head(10)

Translated 1,999 sentences in 496s (4.0 sentences/s on cpu)


,reference,prediction
0,Șeful ONU declară că nu există soluții militar...,eful ONU declară că nu există o soluţie milita...
1,Secretarul General Ban Ki-moon afirmă că răspu...,Secretarul General Ban Ki-moon declară că răsp...
2,Șeful ONU a solicitat din nou tuturor părților...,"eful U.N. a cerut din nou tuturor partidelor, ..."
3,Ban a declarat miercuri în cadrul unei conferi...,Ban a declarat la o conferinţă de presă susţin...
4,Ban și-a exprimat regretul că divizările în co...,El şi-a exprimat regretul că diviziunile din c...
5,Ban le-a cerut celor cinci membri permanenți s...,Ban a cerut celor cinci membri permanenţi să d...
6,8 cifre din sondaje care arată că Donald Trump...,8 Sondaje de opinie care arată că Donald Trump...
7,Unii au încercat să îl eticheteze ca politicia...,Unii au încercat să îl eticheteze un eşec.
8,Alții l-au numit o glumă.,Alţii l-au respins ca pe o glumă.
9,Iar alții așteaptă implozia.,Iar unii se opun unei implozii.


In [7]:
# Side-by-side view of source, model output, and human reference for the first 5 test sentences.
for src, row in zip(dataset.select(range(5)), results.select(range(5))):
    print("EN :", src["translation"]["en"])
    print("MT :", row["prediction"])
    print("REF:", row["reference"])
    print("-" * 100)

EN : UN Chief Says There Is No Military Solution in Syria
MT : eful ONU declară că nu există o soluţie militară în Siria
REF: Șeful ONU declară că nu există soluții militare în Siria
----------------------------------------------------------------------------------------------------
EN : Secretary-General Ban Ki-moon says his response to Russia's stepped up military support for Syria is that "there is no military solution" to the nearly five-year conflict and more weapons will only worsen the violence and misery for millions of people.
MT : Secretarul General Ban Ki-moon declară că răspunsul său la sprijinul militar intensificat al Rusiei pentru Siria este că "nu există o soluţie militară" la conflictul de aproape cinci ani şi că alte arme nu vor face altceva decât să înrăutăţească violenţele şi suferinţa a milioane de oameni.
REF: Secretarul General Ban Ki-moon afirmă că răspunsul său la suportul militar al Rusiei pentru Siria este că „nu există o soluție militară” la conflictul care 

### B.5 Evaluate the quality of translations using the BLEU metric

The reference notebook uses `from evaluate import load` followed by `load("sacrebleu")`. This lab calls SacreBLEU directly with matching defaults: `13a` tokenization, case-sensitive scoring, and exponential smoothing (Post, 2018; SacreBLEU documentation).

1. **Raw corpus BLEU (required primary result):** compare unmodified predictions with the distributed Romanian references. `[references]` is a list containing one complete reference corpus, aligned with the hypotheses.
2. **Diacritic-normalized BLEU (optional diagnostic):** map Romanian comma-below ș/ț and legacy cedilla ş/ţ to the same forms on both sides before rescoring. This measures sensitivity to spelling/Unicode differences. It changes the scoring conditions, so it is reported separately from raw BLEU. The tokenizer check also tests uppercase character coverage; normalization cannot recover a letter already omitted from an output.
3. **Sentence-level BLEU (optional diagnostic):** show the distribution and inspect high- and low-scoring examples. Its mean is not corpus BLEU.
4. **Beam-search comparison and bootstrap intervals (optional extension):** compare greedy decoding with `num_beams=4` on the same data. This extension is not required to complete the assignment's five tasks.

In [8]:
import sacrebleu

hypotheses = results["prediction"]
references = results["reference"]

# 1) Raw corpus BLEU -- greedy decoding, same configuration as the reference notebook
bleu_metric = sacrebleu.BLEU()          # defaults: 13a tokenizer, case-sensitive, exp smoothing
bleu_raw = bleu_metric.corpus_score(hypotheses, [references])
print("Raw BLEU :", bleu_raw)
print("Signature:", bleu_metric.get_signature())
print({"score": bleu_raw.score, "counts": bleu_raw.counts, "totals": bleu_raw.totals,
       "precisions": bleu_raw.precisions, "bp": bleu_raw.bp,
       "sys_len": bleu_raw.sys_len, "ref_len": bleu_raw.ref_len})

# Published result in the reference notebook (same model, data, and decoding; run on a GPU):
REFERENCE_NOTEBOOK_BLEU = 25.18405390123436
print(f"\nReference notebook BLEU: {REFERENCE_NOTEBOOK_BLEU:.2f} | this run: {bleu_raw.score:.2f} "
      f"| difference: {bleu_raw.score - REFERENCE_NOTEBOOK_BLEU:+.2f}")

Raw BLEU : BLEU = 25.18 55.9/31.5/19.2/11.9 (BP = 1.000 ratio = 1.006 hyp_len = 49236 ref_len = 48945)
Signature: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0
{'score': 25.18405390123436, 'counts': [27521, 14902, 8681, 5141], 'totals': [49236, 47237, 45240, 43245], 'precisions': [55.89609229019417, 31.547304020153693, 19.188770999115828, 11.888079546768413], 'bp': 1.0, 'sys_len': 49236, 'ref_len': 48945}

Reference notebook BLEU: 25.18 | this run: 25.18 | difference: +0.00


In [9]:
# Inspect the character mismatch: how often each s/t diacritic form appears in references vs. model
# output, and whether the T5 tokenizer can represent it (encode -> decode round trip without <unk>).
forms = ["ș", "ş", "ț", "ţ", "Ș", "Ş", "Ț", "Ţ"]
print(f"{'char':<6}{'Unicode':<9}{'style':<14}{'in vocab':>9}{'references':>12}{'model output':>14}")
for ch in forms:
    style = "comma-below" if ord(ch) in (0x218, 0x219, 0x21A, 0x21B) else "cedilla"
    in_vocab = tokenizer.decode(tokenizer.encode(ch, add_special_tokens=False)) == ch
    print(f"{ch:<6}U+{ord(ch):04X}   {style:<14}{str(in_vocab):>9}"
          f"{sum(r.count(ch) for r in references):>12,}{sum(h.count(ch) for h in hypotheses):>14,}")
print("\nExample:", tokenizer.tokenize("Șeful ONU"), "->",
      repr(tokenizer.decode(tokenizer.encode("Șeful ONU", add_special_tokens=False))))

char  Unicode  style          in vocab  references  model output


ș     U+0219   comma-below        True       1,882            26


ş     U+015F   cedilla            True           0         1,995
ț     U+021B   comma-below        True       1,753            50


ţ     U+0163   cedilla            True           0         2,379


Ș     U+0218   comma-below       False          40             0
Ş     U+015E   cedilla           False           0             0


Ț     U+021A   comma-below       False           6             0
Ţ     U+0162   cedilla           False           0             0

Example: ['▁', 'Ș', 'e', 'ful', '▁', 'ONU'] -> '<unk>eful ONU'


In [10]:
# 2) Diacritic-normalized BLEU: map comma-below and cedilla variants to the same character.
DIACRITIC_MAP = str.maketrans({"ș": "ş", "Ș": "Ş", "ț": "ţ", "Ț": "Ţ"})

def normalize_ro(text: str) -> str:
    return text.translate(DIACRITIC_MAP)

bleu_norm = sacrebleu.corpus_bleu(
    [normalize_ro(h) for h in hypotheses], [[normalize_ro(r) for r in references]]
)
print("Normalized BLEU:", bleu_norm)

Normalized BLEU: BLEU = 30.23 59.8/36.3/23.9/16.1 (BP = 1.000 ratio = 1.006 hyp_len = 49236 ref_len = 48945)


In [11]:
# 3) Sentence-level BLEU distribution (normalized) and summary table.
sent_bleu = np.array([
    sacrebleu.sentence_bleu(normalize_ro(h), [normalize_ro(r)]).score
    for h, r in zip(hypotheses, references)
])

print(f"{'Metric':<34}{'Raw':>8}{'Normalized':>12}")
print(f"{'Corpus BLEU':<34}{bleu_raw.score:>8.2f}{bleu_norm.score:>12.2f}")
for i, n in enumerate(["1-gram", "2-gram", "3-gram", "4-gram"]):
    print(f"{n + ' precision (%)':<34}{bleu_raw.precisions[i]:>8.1f}{bleu_norm.precisions[i]:>12.1f}")
print(f"{'Brevity penalty':<34}{bleu_raw.bp:>8.3f}{bleu_norm.bp:>12.3f}")
print(f"{'Hyp/ref length ratio':<34}{bleu_raw.sys_len/bleu_raw.ref_len:>8.3f}{bleu_norm.sys_len/bleu_norm.ref_len:>12.3f}")

print("\nSentence-level BLEU (normalized):")
print(f"  mean {sent_bleu.mean():.1f} | median {np.median(sent_bleu):.1f} | "
      f"std {sent_bleu.std():.1f} | share >= 50: {(sent_bleu >= 50).mean():.0%} | "
      f"share < 10: {(sent_bleu < 10).mean():.0%}")

Metric                                 Raw  Normalized
Corpus BLEU                          25.18       30.23
1-gram precision (%)                  55.9        59.8
2-gram precision (%)                  31.5        36.3
3-gram precision (%)                  19.2        23.9
4-gram precision (%)                  11.9        16.1
Brevity penalty                      1.000       1.000
Hyp/ref length ratio                 1.006       1.006

Sentence-level BLEU (normalized):
  mean 27.7 | median 24.4 | std 18.4 | share >= 50: 11% | share < 10: 16%


In [12]:
# Best and worst sentences, to ground the numbers in concrete examples.
order = np.argsort(sent_bleu)
for label, idx in [("BEST", order[-2:][::-1]), ("WORST", order[:2])]:
    for i in idx:
        i = int(i)
        print(f"[{label}] sentence BLEU = {sent_bleu[i]:.1f}")
        print("  EN :", dataset[i]["translation"]["en"])
        print("  MT :", hypotheses[i])
        print("  REF:", references[i])
    print()

[BEST] sentence BLEU = 100.0
  EN : Let us not be hypocrites.
  MT : Să nu fim ipocriţi.
  REF: Să nu fim ipocriți.
[BEST] sentence BLEU = 100.0
  EN : Yes, Chinatown
  MT : Da, Chinatown
  REF: Da, Chinatown

[WORST] sentence BLEU = 0.0
  EN : The great migrations
  MT : Migraţiile mari
  REF: Marile migrații
[WORST] sentence BLEU = 0.0
  EN : Risks
  MT : Riscurile
  REF: Riscuri



In [13]:
# 4) Extension: does beam search beat greedy decoding? Same model, data, and metric; only num_beams changes.
start = time.perf_counter()
results_beam = dataset.map(
    lambda b: translate(b, num_beams=4),
    batched=True, batch_size=BATCH_SIZE,
    remove_columns=dataset.column_names, desc="Translating (beam=4)",
)
beam_secs = time.perf_counter() - start
hyp_beam = results_beam["prediction"]
bleu_beam_raw = sacrebleu.corpus_bleu(hyp_beam, [references])
bleu_beam_norm = sacrebleu.corpus_bleu([normalize_ro(h) for h in hyp_beam],
                                       [[normalize_ro(r) for r in references]])

print(f"{'Decoding':<16}{'Raw BLEU':>10}{'Norm. BLEU':>12}{'BP':>8}{'Time (s)':>10}")
print(f"{'greedy (k=1)':<16}{bleu_raw.score:>10.2f}{bleu_norm.score:>12.2f}{bleu_raw.bp:>8.3f}{elapsed:>10.0f}")
print(f"{'beam (k=4)':<16}{bleu_beam_raw.score:>10.2f}{bleu_beam_norm.score:>12.2f}{bleu_beam_raw.bp:>8.3f}{beam_secs:>10.0f}")


Decoding          Raw BLEU  Norm. BLEU      BP  Time (s)
greedy (k=1)         25.18       30.23   1.000       496
beam (k=4)           25.32       30.37   1.000      4085


In [14]:
# Paired bootstrap resampling (Koehn, 2004): is the beam-vs-greedy BLEU difference significant?
# Corpus BLEU is a function of summed n-gram statistics, so we collect them per sentence once and
# re-aggregate them for each resample of sentence indices.
from sacrebleu.metrics import BLEU

def sentence_stats(hyps, refs):
    rows = []
    for h, r in zip(hyps, refs):
        s = sacrebleu.sentence_bleu(h, [r])
        rows.append(s.counts + s.totals + [s.sys_len, s.ref_len])
    return np.array(rows, dtype=np.int64)            # shape (n, 10)

def bleu_from_stats(stats):
    t = stats.sum(axis=0)
    return BLEU.compute_bleu(correct=t[0:4].tolist(), total=t[4:8].tolist(),
                             sys_len=int(t[8]), ref_len=int(t[9]), smooth_method="exp").score

stats_greedy = sentence_stats(hypotheses, references)
stats_beam = sentence_stats(hyp_beam, references)
# Check: re-aggregated statistics reproduce sacrebleu's corpus scores exactly.
assert abs(bleu_from_stats(stats_greedy) - bleu_raw.score) < 1e-6
assert abs(bleu_from_stats(stats_beam) - bleu_beam_raw.score) < 1e-6

rng = np.random.default_rng(SEED)
n, B = len(references), 1000
diffs, greedy_bs, beam_bs = [], [], []
for _ in range(B):
    idx = rng.integers(0, n, n)
    g, b = bleu_from_stats(stats_greedy[idx]), bleu_from_stats(stats_beam[idx])
    greedy_bs.append(g); beam_bs.append(b); diffs.append(b - g)
diffs = np.array(diffs)
lo, hi = np.percentile(greedy_bs, [2.5, 97.5])
print(f"Greedy BLEU {bleu_raw.score:.2f}  95% CI [{lo:.2f}, {hi:.2f}]")
lo, hi = np.percentile(beam_bs, [2.5, 97.5])
print(f"Beam-4 BLEU {bleu_beam_raw.score:.2f}  95% CI [{lo:.2f}, {hi:.2f}]")
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Beam - greedy: {bleu_beam_raw.score - bleu_raw.score:+.2f}  95% CI [{lo:+.2f}, {hi:+.2f}] | "
      f"share of resamples where beam <= greedy: {(diffs <= 0).mean():.3f}")

Greedy BLEU 25.18  95% CI [24.44, 25.90]
Beam-4 BLEU 25.32  95% CI [24.63, 26.05]
Beam - greedy: +0.14  95% CI [-0.07, +0.35] | share of resamples where beam <= greedy: 0.115


In [15]:
# Save all system outputs next to the source and reference so every number above can be audited.
import pathlib
import pandas as pd

out_dir = pathlib.Path("results"); out_dir.mkdir(exist_ok=True)
table = pd.DataFrame({
    "id": range(len(references)),
    "en": [row["translation"]["en"] for row in dataset],
    "reference_ro": references,
    "greedy_ro": hypotheses,
    "beam4_ro": hyp_beam,
    "greedy_sentence_bleu_norm": sent_bleu.round(2),
})
table.to_csv(out_dir / "translations.csv", index=False, encoding="utf-8")
print(f"saved {len(table):,} rows -> {out_dir / 'translations.csv'}")

saved 1,999 rows -> results\translations.csv


## Part C: Analysis

### C.1 How the commands in Part B work

**B.1 Initialization.** `torch` supplies tensors and model execution; `numpy` supplies numerical operations; `transformers` supplies pre-trained model interfaces. The seed calls initialize the random generators used here. Greedy decoding does not sample, and seeds alone cannot guarantee identical floating-point behavior across devices or releases. `torch.cuda.is_available()` selects a GPU if available; otherwise `torch.device` selects CPU. Configuration constants define the model, English/Romanian keys, task prefix, length limits, batch size, and sample count.

**B.2 Tokenizer and model.** `AutoTokenizer.from_pretrained` loads the vocabulary and tokenization rules; `AutoModelForSeq2SeqLM.from_pretrained` loads the checkpoint configuration and weights from the Hugging Face Hub or cache. In the saved environment, these return `T5Tokenizer` (32,100 entries) and `T5ForConditionalGeneration` (60.5 million parameters). The model's language-model head maps decoder states to vocabulary scores. `.to(device)` places weights on the selected hardware. `model.eval()` disables dropout. The tokenization example shows subword pieces and the appended end-of-sequence ID, `1` (Hugging Face, T5 documentation).

**B.3 Dataset.** `load_dataset("wmt/wmt16", "ro-en", split="test")` loads and caches the official test split: 1,999 dictionaries containing `en` and `ro`. `dataset.select` optionally restricts the sample. The implementation keeps the English inputs and Romanian references aligned. Local copies are supplied in `data/` (Bojar et al., 2016; Hugging Face, Datasets documentation).

**B.4 Translation.** `translate(batch)` takes a batch of translation dictionaries:

1. It prepends `translate English to Romanian: ` to each English sentence. The Romanian references are collected separately and are never passed to `generate`.
2. `tokenizer(..., padding=True, truncation=True, return_tensors="pt")` produces PyTorch token IDs and an attention mask. Padding allows different sentence lengths in one batch; the mask marks padding to ignore. Inputs beyond 1,024 tokens are truncated. `.to(device)` moves the tensors to the same device as the model.
3. `model.generate(..., num_beams=1, do_sample=False, max_length=128)` explicitly disables sampling and uses greedy decoding, stopping at the end-of-sequence token or length cap. `num_beams=4` instead maintains four candidate sequences under the generation scoring rule.
4. `batch_decode(..., skip_special_tokens=True)` converts output IDs to strings and removes special tokens, including padding, end-of-sequence, and unknown tokens.

`@torch.inference_mode()` disables gradient tracking because no weights are updated. `dataset.map(..., batched=True, batch_size=100)` applies the function across the dataset; `remove_columns` removes the original columns while retaining returned `reference` and `prediction` columns. The saved greedy run translated 1,999 sentences in 496 seconds, about four per second.

**B.5 BLEU evaluation.** `sacrebleu.BLEU().corpus_score(hypotheses, [references])` compares one prediction per reference using the `13a` tokenizer. Modified n-gram precision clips each matching hypothesis n-gram count to its reference count, then aggregates counts across sentences for n = 1–4. On SacreBLEU's scale, **BLEU = 100 × BP × exp(¼ Σ log pₙ)**, with precisions expressed as fractions and **BP = min(1, exp(1 − r/c))** for nonempty system output, where c and r are corpus token lengths. The signature `nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0` records one reference, case-sensitive matching, four n-gram orders, tokenization, smoothing, and package version (Papineni et al., 2002; Post, 2018).

`normalize_ro` harmonizes the specified characters on both predictions and references for a separate diagnostic. `sentence_bleu` evaluates individual sentences with smoothing and effective n-gram order. The optional paired bootstrap resamples the same sentence indices for both systems 1,000 times and recomputes corpus BLEU from summed counts. Assertions confirm that these counts reproduce the original corpus scores. The reported fraction of resamples with a nonpositive difference is descriptive, rather than a formal p-value (Koehn, 2004). The final pandas commands save sources, references, both systems' outputs, and sentence scores to `results/translations.csv`.

### C.2 Explanation of the evaluation results (Part B.5)

All results below use the full 1,999-pair test set and one Romanian reference per sentence.

| Setting | BLEU | 1/2/3/4-gram precision (%) | BP | Length ratio |
|---|---|---|---|---|
| Greedy, raw (primary result) | **25.18** | 55.9 / 31.5 / 19.2 / 11.9 | 1.000 | 1.006 |
| Greedy, diacritic-normalized | **30.23** | 59.8 / 36.3 / 23.9 / 16.1 | 1.000 | 1.006 |
| Beam-4, raw | 25.32 | 56.0 / 31.8 / 19.3 / 11.9 | 1.000 | 1.008 |
| Beam-4, diacritic-normalized | 30.37 | 59.9 / 36.5 / 24.1 / 16.1 | 1.000 | 1.008 |

**1. Comparison with the supplied reference.** The recomputed raw greedy BLEU is 25.18405390123436, matching the reference notebook's published score, n-gram match counts (27,521 / 14,902 / 8,681 / 5,141), and hypothesis/reference lengths (49,236 / 48,945). This confirms the same aggregate evaluation result for the saved run; identical scores alone do not prove that all translations or all future runs are identical.

**2. Meaning of the primary score.** BLEU 25.18 measures exact reference overlap under the stated settings. It does **not** mean 25.18% translation accuracy or establish a universal quality grade. Modified precision decreases from 55.9% for unigrams to 11.9% for four-grams, showing fewer exact matches for longer word sequences. Different wording, word order, inflection, spelling, and actual translation errors can all contribute. The corpus length ratio is 1.006 and BP is 1.0, so there is no overall brevity penalty. This does not rule out omissions, additions, or truncation within individual sentences.

**3. Romanian spelling and character coverage.** References contain 1,882 ș and 1,753 ț, while model outputs mostly use cedilla forms (1,995 ş and 2,379 ţ), with only 26 ș and 50 ț. Harmonizing these forms raises BLEU by **5.05 points**, from 25.18 to 30.23, without generating new translations. This is a change in evaluation preprocessing, not a model quality improvement. Raw and normalized results should be compared only with results using the same preprocessing.

The round-trip tokenizer check cannot preserve uppercase Ș/Ş/Ț/Ţ. For example, `Șeful ONU` becomes `<unk>eful ONU`; removing special tokens leaves `eful ONU`. The first prediction therefore omits a real letter. Normalization cannot restore it. Repair would require a model/tokenizer with suitable coverage, or vocabulary expansion with model embedding updates and additional training; fine-tuning alone with the unchanged tokenizer cannot make it emit an unavailable character.

**4. Examples and sentence variability.** Normalized sentence BLEU has mean 27.7, median 24.4, and standard deviation 18.4; approximately 11% score at least 50 and 16% score below 10. These statistics describe individual sentences and must not replace corpus BLEU.

- “Let us not be hypocrites.” → `Să nu fim ipocriţi.` matches the reference after normalization and scores 100.
- “The great migrations” → `Migraţiile mari` scores 0 against `Marile migrații`. The outputs use different inflections and word order while retaining the core meaning, illustrating BLEU's sensitivity on short segments.
- “Risks” → `Riscurile` scores 0 against `Riscuri`; the model adds a definite article (“the risks”). A zero score does not indicate that every aspect of the meaning is wrong.
- In the U.N. example, “parties” becomes `partidelor` (political parties) instead of the contextually intended `părților` (sides), and “to unite” becomes `să unească` without reflexive `se`. These illustrate word-sense and grammatical errors. The model also retains the English abbreviation `U.N.` where the reference uses Romanian `ONU`.

**5. Optional decoding comparison.** Beam-4 increases raw BLEU by only 0.14 points, from 25.18 to 25.32. The paired bootstrap 95% interval for the difference is [−0.07, +0.35], which includes zero and does not establish a reliable improvement. Beam is no better than greedy in 11.5% of resamples; this is a bootstrap fraction, not a formal p-value. Beam took 4,085 seconds versus 496 seconds, about 8.2 times longer in the saved run. These measurements support greedy decoding as a practical baseline here. The comparison is exploratory; decoding choices for a new experiment should be selected on validation data before final test evaluation.

**6. Limitations and improvements.** BLEU rewards exact matches to one reference and does not directly assess meaning or fluency. This test set covers news, so results do not establish performance in other domains. Input/output caps can also affect long sentences. A fuller assessment would inspect adequacy and fluency with human reviewers and use additional references where possible. Better character coverage, a translation-focused checkpoint, or fine-tuning on separate training data could be evaluated in future work. All 1,999 saved outputs remain available for inspection.

## Final Insights

The primary result, **25.18 raw corpus BLEU** on 1,999 sentence pairs, matches the supplied reference notebook's aggregate evaluation. This confirms the baseline result under the stated settings; it does not imply percentage accuracy or guarantee identical results across future environments.

Character handling materially affects evaluation. Harmonizing Romanian cedilla and comma-below forms raises BLEU to **30.23**, without generating new translations. Separately, missing uppercase Romanian character coverage causes actual letter omissions that normalization cannot restore. These findings show why tokenizer inspection and transparent preprocessing matter alongside model selection.

The optional beam-search extension yields only **+0.14 BLEU** while taking approximately **8.2 times longer** in the saved run. Its paired bootstrap interval includes zero, so the evidence does not establish a reliable improvement. The examples also show that reference overlap and preservation of meaning must be assessed together.

## Recommendations

1. **Prioritize Romanian character coverage.** Test uppercase and lowercase diacritics before adopting a model. Evaluate a compatible model/tokenizer with adequate coverage; vocabulary expansion requires corresponding model updates and training. Continue reporting raw and normalized scores separately.
2. **Combine numerical and human assessment.** Retain corpus BLEU with its evaluation signature, then review examples for meaning, agreement, omissions, and named entities. Where feasible, use additional human references to reduce dependence on one wording.
3. **Use greedy decoding as the current baseline.** The observed beam-search gain is small relative to its measured runtime. Compare future decoding settings on validation data and report both translation quality and computational cost.
4. **Keep training, selection, and testing separate.** Use separate training data for any fine-tuning, select models and settings on validation data, and reserve the test split for final evaluation. Document length limits and inspect long outputs for truncation.
5. **Preserve the experiment record.** Keep the notebook's commands, outputs, data identifiers, library versions, metric settings, and saved predictions together. Record hardware and caching conditions when comparing runtime, since these affect timing.

## Conclusion

This lab completes the required English-to-Romanian translation workflow: initializing the libraries, loading a pre-trained tokenizer and model, obtaining a Hugging Face dataset, implementing and applying batch translation, and evaluating predictions with BLEU. The primary score of **25.18** reproduces the reference notebook's aggregate result, while the analysis identifies character coverage, spelling differences, word choice, and grammar as important considerations.

The findings support a disciplined evaluation process that combines reproducible commands, clearly stated scoring conditions, and inspection of meaning and fluency. Future improvements should address the observed character limitations and be tested with separate training, validation, and test data.

## References

- Surdeanu, M., & Valenzuela-Escárcega, M. A. (2024). *Deep Learning for Natural Language Processing: A Gentle Introduction*. Cambridge University Press. Chapter 15 reference notebook: https://github.com/clulab/gentlenlp/blob/main/notebooks/chap15_translation_en_to_ro.ipynb
- Raffel, C., Shazeer, N., Roberts, A., Lee, K., Narang, S., Matena, M., Zhou, Y., Li, W., & Liu, P. J. (2020). Exploring the limits of transfer learning with a unified text-to-text transformer. *Journal of Machine Learning Research, 21*(140), 1–67. https://jmlr.org/papers/v21/20-074.html
- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). Attention is all you need. *Advances in Neural Information Processing Systems, 30*. https://papers.nips.cc/paper/7181-attention-is-all-you-need
- Papineni, K., Roukos, S., Ward, T., & Zhu, W.-J. (2002). BLEU: A method for automatic evaluation of machine translation. *Proceedings of the 40th Annual Meeting of the ACL*, 311–318. https://aclanthology.org/P02-1040/
- Post, M. (2018). A call for clarity in reporting BLEU scores. *Proceedings of the Third Conference on Machine Translation*, 186–191. https://aclanthology.org/W18-6319/
- Koehn, P. (2004). Statistical significance tests for machine translation evaluation. *Proceedings of EMNLP 2004*, 388–395. https://aclanthology.org/W04-3250/
- Bojar, O., et al. (2016). Findings of the 2016 Conference on Machine Translation (WMT16). *Proceedings of the First Conference on Machine Translation*, 131–198. https://aclanthology.org/W16-2301/
- Hugging Face. (n.d.). *T5 documentation*. https://huggingface.co/docs/transformers/model_doc/t5
- Hugging Face. (n.d.). *Datasets: Load*. https://huggingface.co/docs/datasets/loading
- Hugging Face. (n.d.). *WMT16 dataset card*. https://huggingface.co/datasets/wmt/wmt16
- Hugging Face. (n.d.). *T5-small model card*. https://huggingface.co/google-t5/t5-small
- Post, M., et al. (n.d.). *SacreBLEU: Reference BLEU implementation and documentation*. https://github.com/mjpost/sacrebleu